# Baseline - word matching

This notebook explores building a baseline that matches the taxonomi words with the text in the job ads.

There are two versions:
1. The first version matches the untouched skill list from the taxonomi (skill + ev. disambiguation) to the descriptions in the ads.
2. The second version alters the taxonomi, strictly keeping the skill before the comma, and removes the specifications/disambiguations that come after. Example: `"Linux, operativsystem"` turns into `"Linux"`. This allows the matching to the text to catch more hits.

Result:  
The second version collects more skills and will therefore be used as baseline model

In [2]:
import json
import pandas as pd
import re

with open("../data/train_set.json", "r", encoding="utf-8") as f:
    ads_df = pd.DataFrame(json.load(f))

with open("../data/taxonomi_2511.json", "r", encoding="utf-8") as f:
    skills_df = pd.DataFrame(json.load(f))

In [ ]:
# help function to extract skills from text
def extract_skills(text, skills):
    text = text.lower()
    return [
        skill for skill in skills
        if re.search(r"\b" + re.escape(skill.lower()) + r"\b", text) # this allows me to match a skill to an entire word
    ]

## Version 1

In [4]:
skills_full = skills_df["skill"].tolist()

ads_df["baseline_full"] = ads_df["description"].apply(
    lambda text: extract_skills(text, skills_full)
)

In [5]:
baseline_full_skills = ads_df["baseline_full"].explode()
baseline_full_count = baseline_full_skills.value_counts()

## Version 2

In [6]:
# removing specifications/disambiguations from the comma onwards
skills_short = (
    skills_df["skill"]
    .str.split(",").str[0]
    .str.strip()
    .drop_duplicates()
    .tolist()
)

ads_df["baseline_short"] = ads_df["description"].apply(
    lambda text: extract_skills(text, skills_short)
)

In [7]:
baseline_short_skills = ads_df["baseline_short"].explode()
baseline_short_count = baseline_short_skills.value_counts()

## Comparing the two versions

In [8]:
ads_df[["baseline_full", "baseline_short"]]

,baseline_full,baseline_short
0,[],[]
1,[],[SAP]
2,[],[]
3,[],[]
4,[],[]
...,...,...
2601,[],[]
2602,[],[SAP]
2603,[],[]
2604,[],"[Visual Studio, LINQ, ASP.NET MVC]"


In [9]:
comparison = pd.DataFrame({
    "baseline_full": baseline_full_count,
    "baseline_short": baseline_short_count
}).fillna(0)

comparison

,baseline_full,baseline_short
API-design,12.0,12.0
ASP.NET MVC,0.0,3.0
Agila arbetsmetoder,21.0,21.0
Android,0.0,12.0
Apache Kafka,0.0,9.0
...,...,...
Visual Studio Code,0.0,7.0
Webbutveckling,2.0,2.0
Windows,0.0,65.0
Windows Server,0.0,31.0
